# 7. Accuracy, generalization and runtime

Evaluate the exact same labeled crops with each backend. Keep external data out of training and recipe selection. Run timing in fresh processes so previous notebook models do not inflate memory measurements.

Current training/evaluation policy: **literal-string exact only**, including spaces, symbols and case; earliest tied checkpoint wins. TXT permits one framing LF/CRLF only. Old training-selection state is incompatible with resume; historical reference cells retain their original scope. See [literal text policy](../docs/literal_text_policy.md).

## Model selection

Read [0_model_options.ipynb](0_model_options.ipynb) for Base, Tiny and the custom Fast Tiny recipe. Choose `configs/alpr_base.yaml`, `configs/alpr_tiny.yaml`, or `configs/alpr_fast_tiny.yaml` via `CONFIG_PATH`. Keep that choice consistent across notebooks and use separate output folders.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "parseq_ocr").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "parseq_ocr").is_dir():
    raise RuntimeError("Open this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))
from parseq_ocr.config import load_config
# Dummy presets: configs/dummy_base.yaml, dummy_tiny.yaml, dummy_fast_tiny.yaml.
# Real-data presets: configs/alpr_base.yaml, alpr_tiny.yaml, alpr_fast_tiny.yaml.
# PARSEQ_CONFIG overrides this default when it is set.
CONFIG_PATH = os.environ.get("PARSEQ_CONFIG", "configs/dummy_tiny.yaml")
config = load_config(CONFIG_PATH)
print("Config:", CONFIG_PATH)
print("Model:", config["model"], "input:", config["image_size"])
print("Artifacts:", config["output_dir"])


In [ ]:
from parseq_ocr.cli import make_recognizer
from parseq_ocr.evaluation import evaluate_folder

BACKENDS = ["torch", "onnx"]  # Add "tensorrt" after notebook 6 passes on your GPU.
results = {}
for backend in BACKENDS:
    reader = make_recognizer(config, backend)
    results[backend] = evaluate_folder(reader, config["dataset_root"] / "val",
        config["normalization"], batch_size=8,
        output=config["output_dir"] / "evaluation" / backend)
    print(backend, results[backend])
    del reader

Exact matching is literal: no case conversion, whitespace removal, punctuation removal or numeric equivalence. Only complete-string exact counts select checkpoints; earliest ties stay selected. Compare actual native/exported strings rather than assuming parity at reduced precision.

In [ ]:
EXTERNAL_FOLDER = None  # Set to a separate image/TXT folder, including foreign plates.

if EXTERNAL_FOLDER is not None:
    for backend in BACKENDS:
        reader = make_recognizer(config, backend)
        print(backend, evaluate_folder(reader, Path(EXTERNAL_FOLDER), config["normalization"],
            output=config["output_dir"] / "external_evaluation" / backend))
        del reader
else:
    print("External evaluation is optional until you supply an independent labeled folder.")

Public reference results include Brazil, Europe and US crops from OpenALPR. These examples were excluded from our fine-tuning and tuning, but original pretraining exposure cannot be ruled out. Report scores by source/format; unfamiliar alphabets require a compatible trained character head. Data and private fine-tuned weights are not distributed with this repo.

In [ ]:
# Run only after notebook 6, in fresh processes on an otherwise idle GPU.
RUN_TRT_BENCHMARK = False
import subprocess

for batch in ([1, 8, 32] if RUN_TRT_BENCHMARK else []):
    target = config["output_dir"] / "benchmarks" / f"tensorrt_b{batch}.json"
    command = [sys.executable, "-m", "parseq_ocr.cli", "benchmark", "--config",
        str(CONFIG_PATH),
        "--backend", "tensorrt", "--folder", str(config["dataset_root"] / "val"),
        "--batch-size", str(batch), "--repeats", "3", "--output", str(target)]
    subprocess.run(command, cwd=ROOT, check=True)

Timing includes image reading, preprocessing, GPU transfers and text decoding; startup is separate. VRAM is sampled for the child PID and includes its CUDA context. Do not compare isolated numbers with timing gathered while other training or GPU jobs are running. Clear notebook outputs before publishing.